# Activation Space Projections — Experiment Runner

Parameterised notebook for running any of the 30 synthetic-symmetry experiments.

**Pipeline:**
1. Generate dataset
2. Show sample images
3. Train convolutional autoencoder
4. Plot latent activations in 3D using PCA and UMAP (interactive Plotly)

Change the `EXPERIMENT_ID` in the next cell to select which dataset to run.

In [ ]:
# ============================================================
# EXPERIMENT CONFIGURATION — Change these parameters
# ============================================================

EXPERIMENT_ID   = 1        # Which experiment (1-30)
IMAGE_SIZE      = 64       # Image resolution (64 or 256)
N_SAMPLES       = 10000    # Number of images to generate
LATENT_DIM      = 64       # Autoencoder bottleneck dimension
BATCH_SIZE      = 64
NUM_EPOCHS      = 50
LEARNING_RATE   = 1e-3
DROPOUT_RATE    = 0.05

# Output directory for saved plots (Colab virtual drive)
OUTPUT_DIR      = f"outputs/experiment_{EXPERIMENT_ID}"

In [ ]:
# ============================================================
# Install dependencies (Colab)
# ============================================================

!pip install -q umap-learn plotly

In [ ]:
import os, sys, random, io, base64, importlib
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from collections import OrderedDict
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.decomposition import PCA
import umap
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Experiment {EXPERIMENT_ID} | size={IMAGE_SIZE} | n={N_SAMPLES} | latent={LATENT_DIM}")
print(f"Outputs will be saved to: {OUTPUT_DIR}")

---
## 1. Generate Dataset

In [ ]:
# Import the selected experiment's datasets module
exp_path = f"Experiments/{EXPERIMENT_ID}"
sys.path.insert(0, exp_path)

if 'datasets' in sys.modules:
    del sys.modules['datasets']
import datasets as exp_datasets

raw_images, raw_labels = exp_datasets.generate_dataset(n=N_SAMPLES, size=IMAGE_SIZE)
print(f"Dataset shape: {raw_images.shape}  dtype: {raw_images.dtype}  range: [{raw_images.min()}, {raw_images.max()}]")

---
## 2. Show Sample Images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
indices = random.sample(range(len(raw_images)), 10)
for idx, ax in zip(indices, axes.flat):
    ax.imshow(raw_images[idx], cmap='gray')
    ax.axis('off')
fig.suptitle(f'Experiment {EXPERIMENT_ID} — Sample Images ({IMAGE_SIZE}x{IMAGE_SIZE})', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'samples.png'), dpi=120)
plt.show()

---
## 3. Prepare DataLoaders

In [ ]:
# Normalise to [0, 1] and flatten
dataset_flat = raw_images.astype(np.float32) / 255.0
dataset_flat = dataset_flat.reshape(len(dataset_flat), -1)   # (N, IMAGE_SIZE^2)

dataset_tensor = torch.tensor(dataset_flat, dtype=torch.float32)
labels_tensor  = torch.tensor(raw_labels, dtype=torch.long)
full_dataset   = TensorDataset(dataset_tensor, labels_tensor)

train_size = int(0.8 * len(full_dataset))
test_size  = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)}  |  Test: {len(test_dataset)}  |  Batch: {BATCH_SIZE}")

---
## 4. Convolutional Autoencoder

In [ ]:
class ConvAutoencoder(nn.Module):
    """Symmetric Conv Autoencoder — adapts to any power-of-2 image size."""

    def __init__(self, latent_dim, activation=nn.ReLU, n=64,
                 use_batch_norm=True, dropout_rate=0.0):
        super().__init__()
        self.n = n
        self.latent_dim = latent_dim

        # ---------- ENCODER ----------
        enc = []
        channels = [1, 32, 64, 128, 256]
        for i in range(len(channels) - 1):
            enc.append((f'conv{i+1}', nn.Conv2d(channels[i], channels[i+1],
                         kernel_size=3, stride=2, padding=1)))
            if use_batch_norm:
                enc.append((f'bn{i+1}', nn.BatchNorm2d(channels[i+1])))
            enc.append((f'act{i+1}', activation()))
            if dropout_rate > 0:
                enc.append((f'drop{i+1}', nn.Dropout2d(p=dropout_rate)))
        self.conv_encoder = nn.Sequential(OrderedDict(enc))

        # After 4x stride-2 convolutions: spatial = n / 16
        spatial = n // 16
        flattened = 256 * spatial * spatial
        self.flatten = nn.Flatten()
        self.fc_encoder = nn.Linear(flattened, latent_dim)

        # ---------- DECODER ----------
        self.fc_decoder = nn.Linear(latent_dim, flattened)
        self.act_fc     = activation()
        self.unflatten  = nn.Unflatten(1, (256, spatial, spatial))

        dec = []
        dec_ch = list(reversed(channels))   # [256, 128, 64, 32, 1]
        for i in range(len(dec_ch) - 1):
            is_last = (i == len(dec_ch) - 2)
            dec.append((f'deconv{i+1}', nn.ConvTranspose2d(
                dec_ch[i], dec_ch[i+1], kernel_size=3, stride=2,
                padding=1, output_padding=1)))
            if is_last:
                dec.append(('sigmoid', nn.Sigmoid()))
            else:
                if use_batch_norm:
                    dec.append((f'bn_dec{i+1}', nn.BatchNorm2d(dec_ch[i+1])))
                dec.append((f'act_dec{i+1}', activation()))
                if dropout_rate > 0:
                    dec.append((f'drop_dec{i+1}', nn.Dropout2d(p=dropout_rate)))
        self.conv_decoder = nn.Sequential(OrderedDict(dec))

    def encode(self, x):
        if x.dim() == 2:
            x = x.view(-1, 1, self.n, self.n)
        x = self.conv_encoder(x)
        x = self.flatten(x)
        return self.fc_encoder(x)

    def decode(self, latent):
        x = self.act_fc(self.fc_decoder(latent))
        x = self.unflatten(x)
        x = self.conv_decoder(x)
        return x.view(-1, self.n * self.n)

    def forward(self, x):
        return self.decode(self.encode(x))

print("ConvAutoencoder class defined.")

---
## 5. Training

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = ConvAutoencoder(
    latent_dim=LATENT_DIM,
    activation=nn.ReLU,
    n=IMAGE_SIZE,
    use_batch_norm=True,
    dropout_rate=DROPOUT_RATE
).to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_losses = []
test_losses  = []

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    model.train()
    running = 0.0
    for data, _ in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        loss = criterion(model(data), data)
        loss.backward()
        optimizer.step()
        running += loss.item()
    train_losses.append(running / len(train_loader))

    # --- Eval ---
    model.eval()
    running = 0.0
    with torch.no_grad():
        for data, _ in test_loader:
            data = data.to(device)
            running += criterion(model(data), data).item()
    test_losses.append(running / len(test_loader))

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:>3}/{NUM_EPOCHS}]  "
              f"Train: {train_losses[-1]:.4f}  Test: {test_losses[-1]:.4f}")

print("\nTraining complete!")

In [ ]:
# Loss curves
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train')
plt.plot(test_losses,  label='Test')
plt.xlabel('Epoch'); plt.ylabel('BCE Loss')
plt.title('Training & Validation Loss'); plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(OUTPUT_DIR, 'loss_curve.png'), dpi=120)
plt.show()

In [ ]:
# Reconstruction comparison
model.eval()
sample_data, _ = next(iter(test_loader))
sample_data = sample_data.to(device)
with torch.no_grad():
    recon = model(sample_data).cpu().numpy()
orig = sample_data.cpu().numpy()

fig, axes = plt.subplots(2, 10, figsize=(20, 4))
for i in range(10):
    axes[0, i].imshow(orig[i].reshape(IMAGE_SIZE, IMAGE_SIZE), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(recon[i].reshape(IMAGE_SIZE, IMAGE_SIZE), cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('Reconstructed', fontsize=12)
plt.suptitle(f'Experiment {EXPERIMENT_ID} — Reconstructions', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'reconstructions.png'), dpi=120)
plt.show()

---
## 6. Extract Latent Representations

In [ ]:
model.eval()
all_latents = []
all_images  = []

with torch.no_grad():
    for data, _ in test_loader:
        data = data.to(device)
        all_latents.append(model.encode(data).cpu().numpy())
        all_images.append(data.cpu().numpy())

all_latents = np.concatenate(all_latents, axis=0)
all_images  = np.concatenate(all_images,  axis=0)

print(f"Extracted {len(all_latents)} latent vectors of dim {all_latents.shape[1]}")

---
## 7. PCA 3D Projection

In [ ]:
pca = PCA(n_components=3)
pca_3d = pca.fit_transform(all_latents)
print(f"PCA variance explained: {pca.explained_variance_ratio_.sum():.2%}")
print(f"  PC1={pca.explained_variance_ratio_[0]:.2%}  "
      f"PC2={pca.explained_variance_ratio_[1]:.2%}  "
      f"PC3={pca.explained_variance_ratio_[2]:.2%}")

In [ ]:
def build_3d_figure(coords_3d, all_imgs, method_name, axis_labels,
                    title_extra='', img_size=IMAGE_SIZE):
    """
    Build an interactive Plotly 3D scatter with hover-image display.
    Colors are mapped to the x-axis values using Viridis.
    Returns (plotly Figure, full HTML string).
    """

    # --- base64-encode every image for hover ---
    def _to_b64(arr):
        img = Image.fromarray((arr.reshape(img_size, img_size) * 255).astype(np.uint8))
        img = img.resize((128, 128), Image.LANCZOS)
        buf = io.BytesIO()
        img.save(buf, format='PNG')
        return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()

    b64 = [_to_b64(im) for im in all_imgs]

    x, y, z = coords_3d[:, 0], coords_3d[:, 1], coords_3d[:, 2]

    fig = go.Figure(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(
            size=4, opacity=0.7,
            color=x,                   # colour along x-axis
            colorscale='Viridis',
            colorbar=dict(title=axis_labels[0]),
            line=dict(width=0.3, color='white')
        ),
        customdata=np.column_stack([x, y, z, b64]),
        hovertemplate=(
            '<b>Sample #%{pointNumber}</b><br>'
            f'<b>{axis_labels[0]}:</b>' + ' %{customdata[0]}<br>'
            f'<b>{axis_labels[1]}:</b>' + ' %{customdata[1]}<br>'
            f'<b>{axis_labels[2]}:</b>' + ' %{customdata[2]}<br>'
            '<extra></extra>'
        ),
        showlegend=False
    ))

    fig.update_layout(
        title=dict(
            text=f'3D {method_name} of Latent Space '
                 f'({all_latents.shape[1]}D -> 3D){title_extra}',
            x=0.5, xanchor='center', font=dict(size=18)
        ),
        scene=dict(
            xaxis_title=axis_labels[0],
            yaxis_title=axis_labels[1],
            zaxis_title=axis_labels[2],
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.5)),
            bgcolor='#f8f9fa'
        ),
        width=1000, height=800,
        template='plotly_white'
    )

    # Build a standalone HTML page with hover-image panel
    div_id = f'plot3d_{method_name.lower().replace(" ","_")}'
    plot_html = fig.to_html(include_plotlyjs='cdn', div_id=div_id)
    full_html = f"""
<div id="container_{div_id}">
    {plot_html}
    <div id="imgdiv_{div_id}" style="margin-top:20px;text-align:center;min-height:200px;">
        <p style="color:#666">Hover over a point to see its image.</p>
    </div>
</div>
<script>
var p = document.getElementById('{div_id}');
function showImg_{div_id.replace('-','_')}(data) {{
    var pt = data.points[0];
    var cd = pt.customdata;
    document.getElementById('imgdiv_{div_id}').innerHTML =
        '<div style="display:inline-block;padding:20px;background:#f0f0f0;border-radius:12px;">' +
        '<h3>Sample #' + pt.pointNumber + '</h3>' +
        '<img src="' + cd[3] + '" style="border:2px solid #444;border-radius:6px;">' +
        '<div style="margin-top:8px;font-size:13px;">' +
        '{axis_labels[0]}: ' + Number(cd[0]).toFixed(4) + '  |  ' +
        '{axis_labels[1]}: ' + Number(cd[1]).toFixed(4) + '  |  ' +
        '{axis_labels[2]}: ' + Number(cd[2]).toFixed(4) + '</div></div>';
}}
p.on('plotly_hover',  showImg_{div_id.replace('-','_')});
p.on('plotly_click',  showImg_{div_id.replace('-','_')});
</script>
"""
    return fig, full_html

print("build_3d_figure() defined.")

In [ ]:
pca_labels = [
    f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
    f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',
    f'PC3 ({pca.explained_variance_ratio_[2]:.1%})'
]

pca_fig, pca_html = build_3d_figure(
    pca_3d, all_images, 'PCA', pca_labels,
    title_extra=f' | Var={pca.explained_variance_ratio_.sum():.1%}'
)

# Save interactive HTML
pca_path = os.path.join(OUTPUT_DIR, 'pca_3d.html')
with open(pca_path, 'w') as f:
    f.write(pca_html)
print(f"Saved: {pca_path}")

# Display in notebook
display(HTML(pca_html))

---
## 8. UMAP 3D Projection

In [ ]:
print("Running UMAP (default hyperparameters)...")
reducer = umap.UMAP(n_components=3, random_state=42)
umap_3d = reducer.fit_transform(all_latents)
print(f"UMAP projection complete — shape: {umap_3d.shape}")

In [ ]:
umap_labels = ['UMAP1', 'UMAP2', 'UMAP3']

umap_fig, umap_html = build_3d_figure(
    umap_3d, all_images, 'UMAP', umap_labels
)

# Save interactive HTML
umap_path = os.path.join(OUTPUT_DIR, 'umap_3d.html')
with open(umap_path, 'w') as f:
    f.write(umap_html)
print(f"Saved: {umap_path}")

# Display in notebook
display(HTML(umap_html))

---
## 9. Save Data for Offline Visualization

In [ ]:
# Save the raw numerical data so the visualization notebook can reload it
np.savez_compressed(
    os.path.join(OUTPUT_DIR, 'latent_data.npz'),
    latents    = all_latents,
    images     = all_images,
    pca_3d     = pca_3d,
    umap_3d    = umap_3d,
    pca_var    = pca.explained_variance_ratio_,
    experiment = np.array([EXPERIMENT_ID]),
    image_size = np.array([IMAGE_SIZE])
)
print(f"Saved: {os.path.join(OUTPUT_DIR, 'latent_data.npz')}")
print("\nDone!  Open pca_3d.html / umap_3d.html in a browser for interactive 3D plots.")